# Mechanizmus pozornosti v transformeroch

**Predmet:** Hlboké neurónové siete (HNS)

Notebook stavia mechanizmus pozornosti od úplného základu a končí funkčným
GPT-like modelom, ktorý generuje text. Postup je zámerne inkrementálny a každý
krok pridá presne jednu myšlienku a hneď ju overí na dátach.

| Časť | Krok | Čo sa pridá |
|------|------|-------------|
| 1 | dataset Tiny Shakespeare | znaková tokenizácia |
| 2 | bigramový model | východisko bez kontextu |
| 3 | vážená agregácia | ako maticové násobenie mieša minulosť |
| 4 | self-attention | váhy počítané z dát namiesto uniformných |
| 5 | škálovanie $1/\sqrt{d_k}$ | prečo bez neho softmax skolabuje |
| 6 | plný transformer | multi-head, MLP, LayerNorm, rezíduá |

Pokračovanie na obrazové dáta je v notebooku `02_vizualne_transformery.ipynb`.

---

# Ziskanie datasetu

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

## nacitanie

In [67]:
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [68]:
print("Dlzka datasetu je: ", len(text))

Dlzka datasetu je:  1115393


In [69]:
print(text[:200])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


# Zoradenie a zoskupenie pismen v texte

In [70]:
from posixpath import join
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


## "Tokenizator"

In [71]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hi there"))
print(decode(encode("Hii there")))

[46, 47, 1, 58, 46, 43, 56, 43]
Hii there


## Encodovanie celeho datasetu

In [72]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:200])

torch.Size([1115393]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59])


## Rozdelenie na testovacie a trenovacie data

In [73]:
n = int(.9 * len(data))
train_data = data[:n]
val_data = data[n:]

### Rozdelenie na bloky

In [74]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [75]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"Ak je vstup {context}, vystup je {target}")

Ak je vstup tensor([18]), vystup je 47
Ak je vstup tensor([18, 47]), vystup je 56
Ak je vstup tensor([18, 47, 56]), vystup je 57
Ak je vstup tensor([18, 47, 56, 57]), vystup je 58
Ak je vstup tensor([18, 47, 56, 57, 58]), vystup je 1
Ak je vstup tensor([18, 47, 56, 57, 58,  1]), vystup je 15
Ak je vstup tensor([18, 47, 56, 57, 58,  1, 15]), vystup je 47
Ak je vstup tensor([18, 47, 56, 57, 58,  1, 15, 47]), vystup je 58


## "DataLoader"

In [76]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('--------')

for b in range(batch_size):
  for t in range(block_size):
    context = xb[b,:t+1]
    target = yb[b,t]
    print(f"Ak je vstup {context}, vystup je {target}")

inputs:
torch.Size([4, 8])
tensor([[53, 59,  6,  1, 58, 56, 47, 40],
        [49, 43, 43, 54,  1, 47, 58,  1],
        [13, 52, 45, 43, 50, 53,  8,  0],
        [ 1, 39,  1, 46, 53, 59, 57, 43]])
targets:
torch.Size([4, 8])
tensor([[59,  6,  1, 58, 56, 47, 40, 59],
        [43, 43, 54,  1, 47, 58,  1, 58],
        [52, 45, 43, 50, 53,  8,  0, 26],
        [39,  1, 46, 53, 59, 57, 43,  0]])
--------
Ak je vstup tensor([53]), vystup je 59
Ak je vstup tensor([53, 59]), vystup je 6
Ak je vstup tensor([53, 59,  6]), vystup je 1
Ak je vstup tensor([53, 59,  6,  1]), vystup je 58
Ak je vstup tensor([53, 59,  6,  1, 58]), vystup je 56
Ak je vstup tensor([53, 59,  6,  1, 58, 56]), vystup je 47
Ak je vstup tensor([53, 59,  6,  1, 58, 56, 47]), vystup je 40
Ak je vstup tensor([53, 59,  6,  1, 58, 56, 47, 40]), vystup je 59
Ak je vstup tensor([49]), vystup je 43
Ak je vstup tensor([49, 43]), vystup je 43
Ak je vstup tensor([49, 43, 43]), vystup je 54
Ak je vstup tensor([49, 43, 43, 54]), vystup je

# Bigram model

In [77]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1338)

class BigramLanguageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

  def forward(self, idx, targets=None):
    logits = self.token_embedding_table(idx) # (B,T,C) (batch, time, channel)

    if targets is None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):
    # idx is (B, T) aray of indices in the curent context
    for _ in range(max_new_tokens):
      # get prediction
      logits, loss = self(idx)
      # focus on the last time step
      logits = logits[:,-1,:] # become (B, C)
      # aply softmax to get probabilities
      probs = F.softmax(logits, dim=1) # (B, C)
      # sample from the distribution
      idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
      # append sampled index to the running sequence
      idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
    return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 65])
tensor(4.7302, grad_fn=<NllLossBackward0>)

HDDGgDGbUP3?3kpueDyyONp$3
Ao?UbafZ
JmXARz$JGcc3PofAVCb-
;lMHiqCWhFPIysr$J Y$?
AhGoOPT-ZO&bAYh xsrY$R


## Optimizer

In [78]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [83]:
batch_size = 32
for steps in range(20000):

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


2.3654117584228516


In [84]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


ARD:
Toral If-el hthe bus yere,
HAnderas pious chy as bo mbe
ICangr wenge of aund

TI ber t be Ved.
Lach trsiroerome-th lratuto dien, bik's. O,
A:
kes malls styolsoc leane bthe We thy kind y vong fr ncofisele m we sh, hen hthanem cinggheayorayoushano wo rhellaus andsun hancrak
AENotrende d hane CONCHo wid m'Tron,
Ano wat'lair is,
MNo s sh mu then'RDULE:
INDWhie y ppe che bero this wisth'senguprat.
Thesg.
anes ar ceay tithere ham, bee, I menotigean;
An ade conor If no noristo, we,
Killondafethe a


# **self-attenction**

In [86]:
a = torch.tril(torch.ones(3, 3))
b = a / torch.sum(a, 1, keepdim=True)
b

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

In [87]:
# priklad ako pouzit nasobenie matic na "Váženú agregáci"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [88]:
# iný príklad:
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, context_window, dimenzia
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [89]:
# chceme x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)

In [90]:
# version 2: priklad ako pouzit nasobenie matic na "Váženú agregáciu"
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2, atol=1e-7)

True

In [91]:
xbow[0], xbow2[0]

(tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]))

In [92]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [93]:
# version 3: Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3, atol=1e-7)

True

In [94]:
xbow[0], xbow3[0]

(tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]))

In [103]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
# out = wei @ x

out.shape

torch.Size([4, 8, 16])

### Čo sa práve stalo: dopyt, kľúč a hodnota (query, key, value)

Predchádzajúca bunka je jadro celého transformera, preto stojí za rozobratie.

Doteraz boli váhy agregácie **uniformné** a každý predchádzajúci token prispel
rovnako. To je zjavne slabé: pri predpovedaní ďalšieho znaku nie sú všetky
predchádzajúce znaky rovnako dôležité. Self-attention nechá model, aby si váhy
**vypočítal z obsahu**.

Každý token vyšle tri vektory:

| Vektor | Otázka, ktorú zodpovedá |
|--------|--------------------------|
| **query** $q$ | „čo hľadám?“ |
| **key** $k$ | „čo ponúkam?“ |
| **value** $v$ | „čo odovzdám, ak si ma niekto vyberie“ |

Skóre medzi tokenmi $i$ a $j$ je skalárny súčin $q_i \cdot k_j$, teda miera,
nakoľko sa to, čo token $i$ hľadá, zhoduje s tým, čo token $j$ ponúka. Po
maskovaní a softmaxe vznikne rozdelenie, ktorým sa zvážia hodnoty $v$:

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V$$

Všimnite si vo výpise matice `wei` nižšie, že riadky **nie sú** uniformné a
každý token si rozdelil pozornosť inak. Presne o to išlo.

**Prečo maskujeme.** Riadok `wei.masked_fill(tril == 0, float('-inf'))` zabráni
tokenu vidieť budúcnosť. Bez neho by model pri tréningu videl odpoveď a pri
generovaní by zlyhal. Hodnota $-\infty$ sa po softmaxe zmení na nulu.

In [104]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

## A čo ten scale

In [113]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [106]:
k.var()

tensor(1.0449)

In [107]:
q.var()

tensor(1.0700)

In [114]:
wei.var()

tensor(1.1053)

In [115]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [116]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

---

## Typy mechanizmov pozornosti

Teraz sme postavili jeden konkrétny variant: **maskovanú self-attention**.
Tu je zaradený do kontextu ostatných, ktoré sa v praxi používajú.

### 1. Self-attention
$Q$, $K$ aj $V$ vznikajú z **tej istej** sekvencie. Sekvencia spracúva samu seba.

### 2. Maskovaná (kauzálna) self-attention
Self-attention + maska budúcnosti. Nutná pre autoregresívne generovanie,
**toto je variant, ktorý sme implementovali vyššie** (GPT).

### 3. Obojsmerná (nemaskovaná) self-attention
Bez masky, každý token vidí všetky ostatné. Používa ju enkodér BERT, a čo je
pre nás dôležité, **vizuálne transformery**. Pri obraze totiž žiadne „predtým“
a „potom“ neexistuje, obrázok je dostupný celý naraz.

### 4. Krížová pozornosť (cross-attention)
$Q$ pochádza z jednej sekvencie, $K$ a $V$ z **inej**:

```python
q = query(x_dekoder)     # co hladam
k = key(x_enkoder)       # co ponuka druha sekvencia
v = value(x_enkoder)
```

Používa sa v dekodéri strojového prekladu, v modeli DETR na detekciu objektov
a v difúznych modeloch pri podmieňovaní obrázka textom.

### 5. Viachlavová pozornosť (multi-head)
Jedna pozornosť sleduje v podstate jeden typ vzťahu. Multi-head počíta $h$
pozorností paralelne v podpriestoroch dimenzie $d_{\text{model}}/h$ a výsledky
spojí. Rôzne hlavy sa naučia rôzne vzťahy. Cena zostáva rovnaká ako pri jednej
hlave s plnou dimenziou, lebo podpriestory sú úmerne užšie. V kóde nižšie je to
trieda `MultiHeadAttention`.

### 6. Okenná (windowed) pozornosť
Pozornosť obmedzená na lokálne okno, čím zložitosť klesne z $O(n^2)$ na $O(n)$.
Používa ju Swin Transformer, detailne v notebooku 02.

### 7. Kanálová a priestorová pozornosť
Váhy sa nepočítajú zo skalárnych súčinov $Q \cdot K$, ale z agregovaných
štatistík máp príznakov. Sem patrí CBAM, ktorý pridáva pozornosť do konvolučných
sietí (takisto notebook 02).

---

### Zložitosť

Matica skóre má rozmer $n \times n$, takže pamäť aj čas rastú ako $O(n^2 d)$.
Toto obmedzenie je dôvodom, prečo vizuálne transformery nespracúvajú jednotlivé
pixely, ale patche. A prečo vznikla okenná pozornosť. Konkrétne čísla sú
v notebooku 02, časť 1.

# GPT

In [117]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class LanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = LanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.209729 M parameters
step 0: train loss 4.4112, val loss 4.4015
step 100: train loss 2.6576, val loss 2.6632
step 200: train loss 2.5119, val loss 2.5023
step 300: train loss 2.4155, val loss 2.4308
step 400: train loss 2.3514, val loss 2.3670
step 500: train loss 2.3020, val loss 2.3235
step 600: train loss 2.2555, val loss 2.2622
step 700: train loss 2.2139, val loss 2.2241
step 800: train loss 2.1606, val loss 2.1917
step 900: train loss 2.1429, val loss 2.1516
step 1000: train loss 2.1012, val loss 2.1311
step 1100: train loss 2.0642, val loss 2.1148
step 1200: train loss 2.0492, val loss 2.0964
step 1300: train loss 2.0190, val loss 2.0630
step 1400: train loss 2.0015, val loss 2.0471
step 1500: train loss 1.9839, val loss 2.0363
step 1600: train loss 1.9673, val loss 2.0410
step 1700: train loss 1.9519, val loss 2.0299
step 1800: train loss 1.9287, val loss 2.0235
step 1900: train loss 1.9106, val loss 1.9827
step 2000: train loss 1.9093, val loss 1.9966
step 2100: train loss 1.

---

## Zhrnutie

Model má 0,21 M parametrov a po 5000 krokoch klesla validačná strata z 4,40 na
približne 1,83. Vygenerovaný text má správnu **dramatickú štruktúru**: mená
postáv veľkými písmenami, dvojbodky, striedanie replík, interpunkciu. Slová sú
väčšinou vymyslené, čo pri znakovom modeli tejto veľkosti zodpovedá očakávaniu.

Podstatné je, čo tú štruktúru umožnilo: **komunikácia medzi pozíciami cez
pozornosť**. Bigramový model zo začiatku notebooku, ktorý videl len jeden
predchádzajúci znak, nič také nedokázal.

### Čo si odniesť

1. Pozornosť je **vážená agregácia**, kde váhy sa počítajú z dát cez $QK^{\top}$.
2. Škálovanie $1/\sqrt{d_k}$ nie je kozmetika, bez neho softmax skolabuje na one-hot a gradient zanikne.
3. Maska je to jediné, čo robí model autoregresívnym.
4. Blok transformera = **komunikácia** (attention) + **spracovanie** (MLP), obe s rezíduom a normalizáciou.

### Ďalej

Notebook `02_vizualne_transformery.ipynb` zoberie **ten istý blok** a použije ho
na obrázky. Zmenia sa presne dve veci: vstup sa na sekvenciu prevedie cez patch
embedding a **vynechá sa kauzálna maska**.